## PDF Support

Claude can read PDF files. You send the PDF inside a user message, and Claude reads both the text and the images, charts and tables on its pages. You can then ask for a summary, ask questions, or pull out specific information.

In this notebook we send a PDF of the Wikipedia article about Earth (`earth.pdf`) and ask for a one-sentence summary.

### How a PDF is sent

It works like an image. A message's `content` is a **list of blocks**:

1. A `document` block that holds the PDF, sent as **base64** (the file's bytes written as text) with the media type `application/pdf`.
2. A `text` block with your question or instructions.

> **Note:** A request has a maximum size (32 MB at the time of writing) and a page limit, so very large PDFs need to be split or sent another way.

In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-5-5"

**What this cell does:** Loads your API key, creates the client, and picks the model.

- `import base64` is used below to turn the file into text that can be sent in a request.
- `load_dotenv()` reads `ANTHROPIC_API_KEY` from your `.env` file, and `Anthropic()` finds it automatically.
- `model = "claude-haiku-5-5"` is the model used for every request in this notebook.

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    effort=None,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    # Haiku 5.5 thinks adaptively by default; effort sets how hard it thinks
    if effort:
        params["output_config"] = {"effort": effort}

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

**What this cell does:** Defines the helper functions used in the rest of the notebook.

- `add_user_message` and `add_assistant_message` add one message to the conversation. The `content` can be plain text or a list of content blocks, which is how you send a file together with a question.
- `chat` sends the conversation to Claude and returns the whole response. `effort` is optional and sets how hard Claude thinks. `max_tokens` is `4000` because thinking counts toward that limit.
- `text_from_message` returns only the text blocks of a response, skipping any thinking block.

In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

**What this cell does:** Stores a prompt in a variable called `prompt`.

- This is the fire-risk prompt from the images notebook. It is about satellite photos, not PDFs.
- **It is not used in this notebook.** The next cell sends its own question, "Summarize the document in one sentence". You can delete this cell without changing anything.

In [4]:
# TODO: Read image data, feed into Claude
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes
        }
    },
    {
        "type": "text",
        "text": "Summarize the document in one sentence",
         
    }
])
chat_response = chat(messages)

**What this cell does:** Sends the PDF to Claude and asks for a summary.

- `open("earth.pdf", "rb")` reads the file as raw bytes. `base64.standard_b64encode(...).decode("utf-8")` turns them into a base64 text string.
- The user message is a **list of two blocks**. The `document` block carries the PDF, with `"type": "base64"` and `"media_type": "application/pdf"`. The `text` block carries the question.
- The document comes before the question. Putting the file first and the question after it is the usual order.
- `chat(messages)` sends the request, and the whole response is saved in `chat_response`.

In [5]:
print(text_from_message(chat_response))

Earth, the third planet from the Sun and the only known place to harbor life, is an ocean world with a dynamic atmosphere and geology, and this Wikipedia article covers its physical characteristics, etymology, and natural history from its formation about 4.5 billion years ago to the present.


**What this cell does:** Prints Claude's answer.

- `text_from_message(chat_response)` joins the text blocks of the response, so the thinking block is skipped.
- The output is a single sentence that summarises the whole article, as the question asked.
- To use the PDF for something else, change the text in the `text` block. For example, ask a specific question about the document or ask Claude to list the main sections.